In [1]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 97.3 MB/s eta 0:00:00:00:010:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numb

In [2]:
import os
import pandas as pd
from ultralytics import YOLO

# 1. Load your fine-tuned model weights
# (Verify if 'train' matches your active run folder, e.g., 'train', 'train-2', etc.)
model = YOLO('/kaggle/input/models/tahmid76/ta/other/default/1/best_roadvision_model (1).pt')

# 2. Paths to competition files
sample_sub_path = '/kaggle/input/competitions/road-vision/RoadVision_DUET/test/sample_submission.csv'
test_img_dir = '/kaggle/input/competitions/road-vision/RoadVision_DUET/test/images'

image_ids = [os.path.splitext(f)[0] for f in sorted(os.listdir(test_img_dir)) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

submission_data = []

print(f"Processing inference for {len(image_ids)} images...")

for image_id in image_ids:
    # Reconstruct the expected image file path (assuming standard .jpg extension)
    img_path = os.path.join(test_img_dir, f"{image_id}.jpg")
    
    # Double check path if images use a different extension in the folder
    if not os.path.exists(img_path):
        # Fallback check for .png if .jpg isn't present
        img_path = os.path.join(test_img_dir, f"{image_id}.png")

    prediction_list = []
    
    # Only run prediction if the file exists to avoid crashes
    if os.path.exists(img_path):
        results = model.predict(img_path, conf=0.25, verbose=False)
        
        for result in results:
            boxes = result.boxes
            for box in boxes:
                class_id = int(box.cls[0].item())
                confidence = float(box.conf[0].item())
                
                # Extract normalized coordinates (0 to 1)
                xywhn = box.xywhn[0].tolist()
                x_center, y_center, width, height = xywhn
                
                box_string = f"{class_id} {confidence:.4f} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"
                prediction_list.append(box_string)
                
    # CRITICAL FIX: If no vehicles are found, use a single space " " instead of ""
    # This prevents pandas from exporting an empty cell that Kaggle reads as NaN
    if len(prediction_list) == 0:
        combined_prediction_string = " "
    else:
        combined_prediction_string = " ".join(prediction_list)
        
    submission_data.append({
        'image_id': image_id,
        'PredictionString': combined_prediction_string
    })

# 3. Compile, verify, and write out to the workspace
submission_df = pd.DataFrame(submission_data)
submission_df.to_csv('/kaggle/working/submission.csv', index=False)

print("\n🎉 New submission file generated at /kaggle/working/submission.csv")
print(f"Total rows: {len(submission_df)}")
print(f"Number of rows with no detections (safely spaced): {sum(submission_df['PredictionString'] == ' ')}")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Processing inference for 327 images...

🎉 New submission file generated at /kaggle/working/submission.csv
Total rows: 327
Number of rows with no detections (safely spaced): 22
